In [8]:

import pandas as pd

df = pd.read_csv("/content/train.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Column names:", df.columns.tolist())

df.head()


Rows: 891
Columns: 12
Column names: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [9]:

import pandas as pd

def audit_csv(df):
    report = {
        "total_rows": len(df),
        "total_columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "empty_rows": int(df.isna().all(axis=1).sum()),
        "empty_columns": int(df.isna().all(axis=0).sum()),
        "missing_values": df.isna().sum().to_dict(),
        "column_types": df.dtypes.astype(str).to_dict(),
        "invalid_emails": 0,
        "invalid_ages": 0,
    }

    if "Email" in df.columns:
        emails = df["Email"].dropna().astype(str)
        valid = emails.str.fullmatch(
            r"[^@\s]+@[^@\s]+\.[^@\s]+"
        )
        report["invalid_emails"] = int((~valid).sum())

    if "Age" in df.columns:
        ages = pd.to_numeric(df["Age"], errors="coerce")
        invalid = ages.notna() & ~ages.between(0, 120)
        report["invalid_ages"] = int(invalid.sum())

    return report


def display_report(report):
    print("===== CLEANSHEET AUDIT REPORT =====")
    print("Rows:", report["total_rows"])
    print("Columns:", report["total_columns"])
    print("Duplicate rows:", report["duplicate_rows"])
    print("Empty rows:", report["empty_rows"])
    print("Empty columns:", report["empty_columns"])
    print("Missing values:", report["missing_values"])
    print("Column types:", report["column_types"])
    print("Invalid emails:", report["invalid_emails"])
    print("Invalid ages:", report["invalid_ages"])


report = audit_csv(df)
display_report(report)


===== CLEANSHEET AUDIT REPORT =====
Rows: 891
Columns: 12
Duplicate rows: 0
Empty rows: 0
Empty columns: 0
Missing values: {'PassengerId': 0, 'Survived': 0, 'Pclass': 0, 'Name': 0, 'Sex': 0, 'Age': 177, 'SibSp': 0, 'Parch': 0, 'Ticket': 0, 'Fare': 0, 'Cabin': 687, 'Embarked': 2}
Column types: {'PassengerId': 'int64', 'Survived': 'int64', 'Pclass': 'int64', 'Name': 'object', 'Sex': 'object', 'Age': 'float64', 'SibSp': 'int64', 'Parch': 'int64', 'Ticket': 'object', 'Fare': 'float64', 'Cabin': 'object', 'Embarked': 'object'}
Invalid emails: 0
Invalid ages: 0


In [10]:

def missing_value_report(df):
    missing_count = df.isna().sum()
    missing_percentage = (
        missing_count / len(df) * 100
    ).round(2)

    summary = pd.DataFrame({
        "Missing Values": missing_count,
        "Missing Percentage": missing_percentage
    })

    summary = summary[
        summary["Missing Values"] > 0
    ]

    return summary.sort_values(
        by="Missing Percentage",
        ascending=False
    )


missing_report = missing_value_report(df)
display(missing_report)


,Missing Values,Missing Percentage
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22


In [11]:

def generate_quality_warnings(df):
    warnings = []

    # Check for missing values
    for column in df.columns:
        missing_count = int(df[column].isna().sum())

        if missing_count > 0:
            missing_percentage = (
                missing_count / len(df) * 100
            )

            if missing_percentage >= 50:
                severity = "HIGH"
            elif missing_percentage >= 10:
                severity = "MEDIUM"
            else:
                severity = "LOW"

            warnings.append({
                "Column": column,
                "Issue": "Missing values",
                "Count": missing_count,
                "Percentage": round(missing_percentage, 2),
                "Severity": severity
            })

    # Check for duplicate rows
    duplicate_count = int(df.duplicated().sum())

    if duplicate_count > 0:
        warnings.append({
            "Column": "Entire dataset",
            "Issue": "Duplicate rows",
            "Count": duplicate_count,
            "Percentage": round(
                duplicate_count / len(df) * 100, 2
            ),
            "Severity": "MEDIUM"
        })

    return pd.DataFrame(warnings)


warnings_df = generate_quality_warnings(df)

if warnings_df.empty:
    print("No data-quality issues detected.")
else:
    display(warnings_df)


,Column,Issue,Count,Percentage,Severity
0,Age,Missing values,177,19.87,MEDIUM
1,Cabin,Missing values,687,77.10,HIGH
2,Embarked,Missing values,2,0.22,LOW


In [12]:

# Export the quality warnings to a CSV file

output_file = "/content/cleansheet_quality_report.csv"

warnings_df.to_csv(output_file, index=False)

print("Report exported successfully!")
print("File location:", output_file)

# Preview the exported report
exported_report = pd.read_csv(output_file)
display(exported_report)


Report exported successfully!
File location: /content/cleansheet_quality_report.csv


,Column,Issue,Count,Percentage,Severity
0,Age,Missing values,177,19.87,MEDIUM
1,Cabin,Missing values,687,77.10,HIGH
2,Embarked,Missing values,2,0.22,LOW


In [13]:

# Create a small dataset with intentional quality issues

test_df = pd.DataFrame({
    "Name": ["Anu", "Ravi", "Anu", "Maya", "John"],
    "Age": [20, -5, 20, None, 150],
    "Email": [
        "anu@example.com",
        "ravi.example.com",
        "anu@example.com",
        None,
        "john@example.com"
    ]
})

print("TEST DATASET:")
display(test_df)

print("\nBASIC AUDIT:")
test_report = audit_csv(test_df)
display_report(test_report)

print("\nQUALITY WARNINGS:")
test_warnings = generate_quality_warnings(test_df)
display(test_warnings)


TEST DATASET:


,Name,Age,Email
0,Anu,20.0,anu@example.com
1,Ravi,-5.0,ravi.example.com
2,Anu,20.0,anu@example.com
3,Maya,NaN,None
4,John,150.0,john@example.com



BASIC AUDIT:
===== CLEANSHEET AUDIT REPORT =====
Rows: 5
Columns: 3
Duplicate rows: 1
Empty rows: 0
Empty columns: 0
Missing values: {'Name': 0, 'Age': 1, 'Email': 1}
Column types: {'Name': 'object', 'Age': 'float64', 'Email': 'object'}
Invalid emails: 1
Invalid ages: 2

QUALITY WARNINGS:


,Column,Issue,Count,Percentage,Severity
0,Age,Missing values,1,20.0,MEDIUM
1,Email,Missing values,1,20.0,MEDIUM
2,Entire dataset,Duplicate rows,1,20.0,MEDIUM


In [14]:

def generate_quality_warnings(df):
    warnings = []

    def add_warning(column, issue, count, percentage, severity):
        warnings.append({
            "Column": column,
            "Issue": issue,
            "Count": count,
            "Percentage": round(percentage, 2),
            "Severity": severity
        })

    def get_severity(percentage):
        if percentage >= 50:
            return "HIGH"
        elif percentage >= 10:
            return "MEDIUM"
        return "LOW"

    # 1. Missing values
    for column in df.columns:
        count = int(df[column].isna().sum())

        if count > 0:
            percentage = count / len(df) * 100
            add_warning(
                column, "Missing values", count,
                percentage, get_severity(percentage)
            )

    # 2. Duplicate rows
    duplicates = int(df.duplicated().sum())

    if duplicates > 0:
        percentage = duplicates / len(df) * 100
        add_warning(
            "Entire dataset", "Duplicate rows", duplicates,
            percentage, get_severity(percentage)
        )

    # 3. Invalid ages
    if "Age" in df.columns:
        ages = pd.to_numeric(df["Age"], errors="coerce")
        invalid = ages.notna() & ~ages.between(0, 120)
        count = int(invalid.sum())

        if count > 0:
            percentage = count / len(df) * 100
            add_warning(
                "Age", "Invalid ages", count,
                percentage, get_severity(percentage)
            )

    # 4. Invalid email addresses
    if "Email" in df.columns:
        emails = df["Email"].dropna().astype(str)
        valid = emails.str.fullmatch(
            r"[^@\s]+@[^@\s]+\.[^@\s]+"
        )
        count = int((~valid).sum())

        if count > 0:
            percentage = count / len(df) * 100
            add_warning(
                "Email", "Invalid emails", count,
                percentage, get_severity(percentage)
            )

    return pd.DataFrame(
        warnings,
        columns=[
            "Column", "Issue", "Count",
            "Percentage", "Severity"
        ]
    )


In [15]:

test_warnings = generate_quality_warnings(test_df)
display(test_warnings)


,Column,Issue,Count,Percentage,Severity
0,Age,Missing values,1,20.0,MEDIUM
1,Email,Missing values,1,20.0,MEDIUM
2,Entire dataset,Duplicate rows,1,20.0,MEDIUM
3,Age,Invalid ages,2,40.0,MEDIUM
4,Email,Invalid emails,1,20.0,MEDIUM


In [16]:

print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())


Dataset shape: (891, 12)

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Duplicate rows: 0


In [17]:

print("Synthetic test shape:", test_df.shape)
print("Duplicate rows:", test_df.duplicated().sum())

ages = pd.to_numeric(test_df["Age"], errors="coerce")
print("Invalid ages:", int((ages.notna() & ~ages.between(0, 120)).sum()))

emails = test_df["Email"].dropna().astype(str)
valid_emails = emails.str.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+")
print("Invalid emails:", int((~valid_emails).sum()))


Synthetic test shape: (5, 3)
Duplicate rows: 1
Invalid ages: 2
Invalid emails: 1
